# BERT: Words in Context

BERT stands for **Bidirectional Encoder Representations from Transformers**. It uses an encoder architecture that allows each token to attend to the surrounding tokens on both sides.

BERT was pretrained using masked language modeling, where selected tokens are hidden and the model learns to predict them from context. Here, we use its contextual representations directly, without masking words.

## Same word, different contexts

In the NLP notebook, “orange” received the same token ID whether it referred to a fruit or a color. Here, we examine how BERT represents it in different sentences:

- “She peeled an orange for breakfast.”
- “He sliced an orange into wedges.”
- “She painted the door orange.”
- “He wore an orange jacket.”

The token ID stays the same, but BERT produces a contextual vector for each occurrence. We will compare those vectors to explore how the surrounding words affect the representation.

In [1]:
import torch
import pandas as pd
from IPython.display import Markdown, display
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

phrases = [
    "She peeled an orange for breakfast.",
    "He sliced an orange into wedges.",
    "She painted the door orange.",
    "He wore an orange jacket.",
]

inputs = tokenizer(phrases, padding=True, return_tensors="pt")

with torch.inference_mode():
    hidden_states = model(**inputs).last_hidden_state

orange_token_id = tokenizer.convert_tokens_to_ids("orange")
orange_embeddings = []

for phrase_index, token_ids in enumerate(inputs["input_ids"]):
    orange_position = (token_ids == orange_token_id).nonzero(as_tuple=True)[0].item()
    orange_embeddings.append(hidden_states[phrase_index, orange_position])

embeddings = torch.stack(orange_embeddings)

display(Markdown("### Contextual representations of “orange”"))

pd.DataFrame(
    embeddings[:, :8].numpy(),
    index=phrases,
    columns=[f"Dimension {i}" for i in range(1, 9)],
).round(3)

/home/nick/github-projects/prompting-base-models/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2002.97it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	ca

### Contextual representations of “orange”

,Dimension 1,Dimension 2,Dimension 3,Dimension 4,Dimension 5,Dimension 6,Dimension 7,Dimension 8
She peeled an orange for breakfast.,0.358,0.027,0.024,-0.112,0.588,0.748,-0.001,0.712
He sliced an orange into wedges.,0.617,-0.262,-0.440,0.174,0.278,0.577,-0.632,1.005
She painted the door orange.,0.164,-0.455,-0.557,-0.122,0.471,0.471,0.383,1.365
He wore an orange jacket.,-0.155,-0.640,-0.498,-0.135,0.566,0.394,-0.294,0.500


### About the output

Each row shows the first eight dimensions of BERT’s 768-dimensional contextual vector for `orange` in one sentence.

The token has the same ID in every sentence, but its vector changes with the surrounding text. Individual dimensions do not have simple labels such as “fruit” or “color”; we compare the full vectors to examine their similarity.

## Comparing contextual embeddings

We use cosine similarity to compare the full vectors for `orange` across the four sentences.

A higher score means the vectors point in more similar directions. We can inspect whether the two fruit examples are closer to each other than to the color examples, and vice versa.

This is a small illustration of contextual similarity, not a classification benchmark. Sentence structure and surrounding vocabulary can also affect the scores.

In [2]:
similarities = torch.nn.functional.cosine_similarity(
    embeddings.unsqueeze(1),
    embeddings.unsqueeze(0),
    dim=2,
)

labels = ["Fruit: breakfast", "Fruit: wedges", "Color: door", "Color: jacket"]

pd.DataFrame(
    similarities.numpy(),
    index=labels,
    columns=labels,
).round(3)

,Fruit: breakfast,Fruit: wedges,Color: door,Color: jacket
Fruit: breakfast,1.000,0.827,0.611,0.572
Fruit: wedges,0.827,1.000,0.564,0.534
Color: door,0.611,0.564,1.000,0.664
Color: jacket,0.572,0.534,0.664,1.000


### Interpreting the similarities

Each row and column represents one occurrence of `orange`. Every diagonal value is `1.000` because a vector is being compared with itself.

Compare the two fruit examples, then the two color examples, and finally the fruit–color pairs. Higher scores indicate more similar contextual vectors, but they are not probabilities or confidence scores.

The scores reflect the surrounding sentences as well as the word’s meaning. With only four examples, this is an illustration of BERT’s contextual representations rather than evidence of reliable word-sense classification.